In [1]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_tavily import TavilySearch
from dotenv import load_dotenv
load_dotenv()

search_tool = TavilySearch(
    max_results=5,
    topic = "general"
)

In [2]:
search_tool.invoke("真香是什么梗")

{'query': '真香是什么梗',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.sohu.com/a/273050503_535979',
   'title': '真香是什么意思？真香定律什么意思？什么梗？来源于哪？都在这了_事情',
   'content': '# 真香是什么意思？真香定律什么意思？什么梗？来源于哪？都在这了. 不得不说，网络用语实在是让我们不知道什么段子突然就火了。比如最近特别火的真香定律。就给大家说一下来源于哪，什么意思！. 这个是来源于变形计的其中一期，参加的变形的孩子是王境泽。在拍摄的过程中，由于是在农村，刚开始的时候，他并不喜欢这个环境，并且说：我王境泽就是饿死，死外面，从这跳下去，不会吃你们一点东西！”，后面又端着碗边吃边感慨“真香”，在当时的时候，虽然这个被网友截图发到网上，但是没有特别火，直到今天，彻底让真香火起来了。. 比如很多人说我要克制我自己，这个时候不要再从网上买东西了，但是这边说完，就开始又开始网购了。很多人都，我们做事情都逃不过真香定律的，因为你自己都不知道你最不愿意做的事情，或者信誓旦旦不会去做的事情突然就因为某些事情去做了。. 在明白了什么意思之后，给大家说说18年火的原因吧，在前段时间的时候作为表情包的三大巨头：有钱真的可以为所欲为、打工是不可能打工的、真香、被很多网友开始制作成表情包了，又纷纷开始使用，从那个时候开始，真香定律开始慢慢火了。. 而又被大众熟知是另外一件事情，请段时间理发的小吴因为眉毛突然就火了，在接受采访的时候说自己不会进军娱乐圈，不会拍广告，结果刚说完，不到一个月，各种代言广告就接踵而来了。. 同样的，这个事情以后，还有朴树，还上了热搜，说自己不喜欢录制节目，也不喜欢坐摩托车，结果说完，到坐完摩托车以后，还给导演说：如果要是有山路的话，更爽。. 目前的网络用语更新的速度也很快，如果稍不注意，就多了很多段子，自己都不知道，看来要与时俱进，真的是难度挺大呀！返回搜狐，查看更多.',
   'score': 0.7927375,
   'raw_content': None},
  {'url': 'https://baike.baidu.com/

In [3]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model = "deepseek-v4-pro",
    tools = [search_tool],
    system_prompt = "你是一个智能助手，你使用工具来解决用户问题"
)

In [29]:
response = agent.invoke(
    {"messages": [HumanMessage(content="真相是什么梗？")]},
)
for message in response['messages']:
    message.pretty_print()

KeyboardInterrupt: 

In [4]:
from langchain_core.tools import tool

tavily = TavilySearch(
    max_results=5,
    topic = "general"
)

@tool
def web_search(query: str):
    """Search web pages for information"""
    return tavily.invoke(query)

In [5]:
from pydantic import BaseModel, Field

class Reference(BaseModel):
    title: str = Field(description="The title of the web pages cited in the answer")
    url: str = Field(description="The url of the web pages cited in the answer")

class AnswerInfo(BaseModel):
    answer: str = Field(description="The final answer of user")
    reference: list[Reference] = Field(description="The web pages cited in the answer")

In [16]:
from langchain.chat_models import init_chat_model
from langchain.agents.structured_output import ToolStrategy

llm = init_chat_model(
    "deepseek-v4-pro",
    extra_body={"thinking": {"type": "disabled"}}  # 关闭思考模式
)

agent = create_agent(
    model = llm,
    tools=[web_search],
    system_prompt="""你是一个智能助手，使用工具解决问题。
当你收集到足够信息后，必须用以下JSON格式输出最终答案（不要包含其他文字，只输出纯JSON）：
{"answer": "你的最终回答", "reference": [{"title": "引用的网页标题", "url": "引用的网页URL"}]}""",
    response_format=ToolStrategy(AnswerInfo)
)

In [ ]:
import json

response = agent.invoke(
    {"messages": [HumanMessage(content="真香是什么梗？")]}
)

print(response)

In [14]:
print(response["messages"][-1].content)

Returning structured response: answer='"真香"是一个网络流行梗，源自湖南卫视综艺节目《变形计》第八季《远山的抉择》（2014年播出）。节目中，城市少年王境泽刚到农村时非常抵触，放出狠话："我王境泽就是饿死，死外面，从这跳下去，不会吃你们一点东西！"结果没过多久，他就端起碗大口吃饭，并感叹"真香！"。\n\n这个强烈的反差场景被网友截图并制作成表情包，后来演变成"真香定律"，用来调侃那些一开始信誓旦旦拒绝某件事、最后却主动打脸接受的反转行为。比如"我再也不网购了——真香"、"我绝不进娱乐圈——真香"等。' reference=[Reference(title='真香是什么意思？真香定律什么意思？什么梗？来源于哪？都在这了', url='https://www.sohu.com/a/273050503_535979'), Reference(title='真香（网络流行语） - 百度百科', url='https://baike.baidu.com/item/%E7%9C%9F%E9%A6%99/22700736'), Reference(title='真香是什么梗？ - 知乎', url='https://www.zhihu.com/question/291067203')]
